In [2]:
from pathlib import Path
import duckdb
import time


# ---------------------------------------------------------
# Пути
# ---------------------------------------------------------

PROJECT_ROOT = Path("..")

MART_EVENTS = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_events.parquet"
)

MART_QUESTIONS = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_questions.parquet"
)

BUNDLE_ELAPSED = (
    PROJECT_ROOT
    / "data"
    / "mart_cache"
    / "mart_questions"
    / "question_bundle_elapsed.parquet"
)


# ---------------------------------------------------------
# Пороги
# ---------------------------------------------------------

FAST_THRESHOLD_MS = 15017.84
SLOW_THRESHOLD_MS = 27187.37


# ---------------------------------------------------------
# Проверяем файлы
# ---------------------------------------------------------

for path in [
    MART_EVENTS,
    MART_QUESTIONS,
    BUNDLE_ELAPSED,
]:
    print(path, "→", path.exists())


con = duckdb.connect()


# ---------------------------------------------------------
# Считаем FAST/SLOW метрики
# ---------------------------------------------------------

started = time.perf_counter()


fast_slow_metrics = con.execute(f"""

    SELECT

        e.question_id,


        -- =================================================
        -- FAST
        -- =================================================

        SUM(
            CASE
                WHEN
                    e.answered_correctly = 1
                    AND b.elapsed_time > 0
                    AND b.elapsed_time <= {FAST_THRESHOLD_MS}
                THEN 1
                ELSE 0
            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE
                    b.elapsed_time > 0
                    AND b.elapsed_time <= {FAST_THRESHOLD_MS}
            ),

            0

        ) AS fast_correct_share,


        SUM(
            CASE
                WHEN
                    e.answered_correctly = 0
                    AND b.elapsed_time > 0
                    AND b.elapsed_time <= {FAST_THRESHOLD_MS}
                THEN 1
                ELSE 0
            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE
                    b.elapsed_time > 0
                    AND b.elapsed_time <= {FAST_THRESHOLD_MS}
            ),

            0

        ) AS fast_incorrect_share,


        -- =================================================
        -- SLOW
        -- =================================================

        SUM(
            CASE
                WHEN
                    e.answered_correctly = 1
                    AND b.elapsed_time >= {SLOW_THRESHOLD_MS}
                THEN 1
                ELSE 0
            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE
                    b.elapsed_time > 0
                    AND b.elapsed_time >= {SLOW_THRESHOLD_MS}
            ),

            0

        ) AS slow_correct_share,


        SUM(
            CASE
                WHEN
                    e.answered_correctly = 0
                    AND b.elapsed_time >= {SLOW_THRESHOLD_MS}
                THEN 1
                ELSE 0
            END
        ) * 1.0

        /

        NULLIF(

            COUNT(*) FILTER (
                WHERE
                    b.elapsed_time > 0
                    AND b.elapsed_time >= {SLOW_THRESHOLD_MS}
            ),

            0

        ) AS slow_incorrect_share


    FROM read_parquet('{MART_EVENTS}') e


    LEFT JOIN read_parquet('{BUNDLE_ELAPSED}') b

        ON
            e.user_id = b.user_id

        AND
            e.task_container_id = b.task_container_id


    WHERE

        e.content_type_id = 0

        AND e.question_id IS NOT NULL

        AND e.answered_correctly IN (0, 1)


    GROUP BY
        e.question_id

""").df()


elapsed = time.perf_counter() - started


print(
    f"Метрики рассчитаны за {elapsed / 60:.2f} min"
)

print(
    f"Вопросов: {len(fast_slow_metrics):,}"
)

../data/processed/mart_events.parquet → True
../data/processed/mart_questions.parquet → True
../data/mart_cache/mart_questions/question_bundle_elapsed.parquet → True
Метрики рассчитаны за 0.24 min
Вопросов: 13,523


In [3]:
# ---------------------------------------------------------
# Загружаем mart_questions
# ---------------------------------------------------------

questions = con.execute(f"""
    SELECT *
    FROM read_parquet('{MART_QUESTIONS}')
""").df()


print("Исходный mart_questions:")
print(questions.shape)


# ---------------------------------------------------------
# Добавляем новые метрики
# ---------------------------------------------------------

questions = questions.merge(
    fast_slow_metrics,
    on="question_id",
    how="left",
    validate="one_to_one"
)


print("\nПосле добавления:")
print(questions.shape)


# ---------------------------------------------------------
# Проверяем результат
# ---------------------------------------------------------

new_columns = [
    "fast_correct_share",
    "fast_incorrect_share",
    "slow_correct_share",
    "slow_incorrect_share",
]

print("\nНовые поля:")
print(questions[new_columns].head())

Исходный mart_questions:
(13523, 20)

После добавления:
(13523, 24)

Новые поля:
   fast_correct_share  fast_incorrect_share  slow_correct_share  \
0            0.959585              0.040415            0.721116   
1            0.967128              0.032872            0.668161   
2            0.894640              0.105360            0.427254   
3            0.914674              0.085326            0.579634   
4            0.646268              0.353732            0.459724   

   slow_incorrect_share  
0              0.278884  
1              0.331839  
2              0.572746  
3              0.420366  
4              0.540276  


In [4]:
import numpy as np


questions["fast_sum"] = (
    questions["fast_correct_share"]
    + questions["fast_incorrect_share"]
)

questions["slow_sum"] = (
    questions["slow_correct_share"]
    + questions["slow_incorrect_share"]
)


print("FAST:")
print(
    questions["fast_sum"]
    .dropna()
    .describe()
)


print("\nSLOW:")
print(
    questions["slow_sum"]
    .dropna()
    .describe()
)

FAST:
count    13472.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: fast_sum, dtype: float64

SLOW:
count    13507.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: slow_sum, dtype: float64


In [5]:
OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_questions_with_time_errors.parquet"
)


questions.drop(
    columns=["fast_sum", "slow_sum"],
    errors="ignore"
).to_parquet(
    OUTPUT,
    index=False
)


print(f"✅ Сохранено: {OUTPUT}")

✅ Сохранено: ../data/processed/mart_questions_with_time_errors.parquet
